In [19]:
!pip install tensorflow numpy

In [2]:
# 1. KoNLPy Okt 형태소 분석기 사용

# !pip install konlpy
from konlpy.tag import Okt

okt = Okt()

text = "나는 자연어 처리를 배운다"

# 토큰화
token = okt.morphs(text)
print(token)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 27.2 MB/s eta 0:00:00
['나', '는', '자연어', '처리', '를', '배운다']


In [3]:
# 2. 각 토큰에 고유 인덱스 부여

word2index = {}

for voca in token:
    if voca not in word2index:
        word2index[voca] = len(word2index)

print(word2index)

{'나': 0, '는': 1, '자연어': 2, '처리': 3, '를': 4, '배운다': 5}


In [4]:
# 3. 직접 원-핫 인코딩 함수 만들기


def one_hot_encoding(word, word2index):
    one_hot_vector = [0] * len(word2index)
    index = word2index[word]
    one_hot_vector[index] = 1
    return one_hot_vector

print(one_hot_encoding("자연어", word2index))

[0, 0, 1, 0, 0, 0]


### Keras Tokenizer를 이용한 정수 인코딩 + 원-핫 인코딩

In [5]:
# 4. Keras Tokenizer 정수 인코딩

!pip install tensorflow.keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

text = "나랑 점심 먹으러 갈래 점심 메뉴는 햄버거 갈래 갈래 햄버거 최고야"

t = Tokenizer()
t.fit_on_texts([text])

print(t.word_index)

{'갈래': 1, '점심': 2, '햄버거': 3, '나랑': 4, '먹으러': 5, '메뉴는': 6, '최고야': 7}


In [6]:
# 5. texts_to_sequences()로 정수 시퀀스 변환


sub_text = "점심 먹으러 갈래 메뉴는 햄버거 최고야"

encoded = t.texts_to_sequences([sub_text])[0]

print(encoded)

[2, 5, 1, 6, 3, 7]


In [7]:
# 6. to_categorical()로 원-핫 인코딩


one_hot = to_categorical(encoded)

print(one_hot)

[[0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]]


#### RNN을 이용한 한국어 텍스트 생성

In [18]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN


# 재현성을 위한 seed 설정
np.random.seed(42)
tf.random.set_seed(42)


# 1. 학습 데이터 준비

text = """고기는 씹어야 맛이고, 말은 해야 맛이라
낫말은 새가 듣고 밤말은 쥐가 듣는다
구슬이 서 말이라도 꿰어야 보배"""


# 2. 토크나이저 생성 및 단어 집합 크기 확인


t = Tokenizer()
t.fit_on_texts([text])

vocab_size = len(t.word_index) + 1
# 케라스 토크나이저의 정수 인코딩은 인덱스가 1부터 시작하지만,
# 케라스 원-핫 인코딩에서 배열의 인덱스가 0부터 시작하기 때문에
# 배열의 크기를 실제 단어 집합의 크기보다 +1로 생성해야하므로 미리 +1 선언

print("단어 집합의 크기 :", vocab_size)
print(t.word_index)

단어 집합의 크기 : 18
{'고기는': 1, '씹어야': 2, '맛이고': 3, '말은': 4, '해야': 5, '맛이라': 6, '낫말은': 7, '새가': 8, '듣고': 9, '밤말은': 10, '쥐가': 11, '듣는다': 12, '구슬이': 13, '서': 14, '말이라도': 15, '꿰어야': 16, '보배': 17}


In [9]:
# 3. 훈련 데이터 생성


sequences = []

for line in text.split("\n"):
    encoded = t.texts_to_sequences([line])[0]

    for i in range(1, len(encoded)):
        sequence = encoded[:i + 1]
        sequences.append(sequence)

print("학습에 사용할 샘플의 개수 :", len(sequences))
print(sequences)

학습에 사용할 샘플의 개수 : 14
[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6], [7, 8], [7, 8, 9], [7, 8, 9, 10], [7, 8, 9, 10, 11], [7, 8, 9, 10, 11, 12], [13, 14], [13, 14, 15], [13, 14, 15, 16], [13, 14, 15, 16, 17]]


In [10]:
# 4. 가장 긴 샘플 길이에 맞춰 패딩



max_len = max(len(sequence) for sequence in sequences)

print("샘플의 최대 길이 :", max_len)

sequences = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="pre"
)

print(sequences)

샘플의 최대 길이 : 6
[[ 0  0  0  0  1  2]
 [ 0  0  0  1  2  3]
 [ 0  0  1  2  3  4]
 [ 0  1  2  3  4  5]
 [ 1  2  3  4  5  6]
 [ 0  0  0  0  7  8]
 [ 0  0  0  7  8  9]
 [ 0  0  7  8  9 10]
 [ 0  7  8  9 10 11]
 [ 7  8  9 10 11 12]
 [ 0  0  0  0 13 14]
 [ 0  0  0 13 14 15]
 [ 0  0 13 14 15 16]
 [ 0 13 14 15 16 17]]


In [11]:
# 5. 입력 X와 정답 y 분리


sequences = np.array(sequences)

X = sequences[:, :-1]
y = sequences[:, -1]

print("X:")
print(X)

print("y:")
print(y)

X:
[[ 0  0  0  0  1]
 [ 0  0  0  1  2]
 [ 0  0  1  2  3]
 [ 0  1  2  3  4]
 [ 1  2  3  4  5]
 [ 0  0  0  0  7]
 [ 0  0  0  7  8]
 [ 0  0  7  8  9]
 [ 0  7  8  9 10]
 [ 7  8  9 10 11]
 [ 0  0  0  0 13]
 [ 0  0  0 13 14]
 [ 0  0 13 14 15]
 [ 0 13 14 15 16]]
y:
[ 2  3  4  5  6  8  9 10 11 12 14 15 16 17]


In [13]:
# 6. y 원-핫 인코딩

y = to_categorical(y, num_classes=vocab_size)

print(y)
print("y shape:", y.shape)

[[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
y shape: (14, 18)


In [14]:
# 7. RNN 모델 생성


model = Sequential()

model.add(Embedding(
    input_dim=vocab_size,
    output_dim=10,
    input_length=max_len - 1
))

model.add(SimpleRNN(32))

model.add(Dense(
    vocab_size,
    activation="softmax"
))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
# 8. 모델 학습


model.fit(
    X,
    y,
    epochs=200,
    verbose=2
)

Epoch 1/200
1/1 - 2s - 2s/step - accuracy: 0.0714 - loss: 2.8757
Epoch 2/200
1/1 - 0s - 50ms/step - accuracy: 0.0714 - loss: 2.8672
Epoch 3/200
1/1 - 0s - 46ms/step - accuracy: 0.0714 - loss: 2.8587
Epoch 4/200
1/1 - 0s - 49ms/step - accuracy: 0.2143 - loss: 2.8501
Epoch 5/200
1/1 - 0s - 46ms/step - accuracy: 0.2143 - loss: 2.8414
Epoch 6/200
1/1 - 0s - 447ms/step - accuracy: 0.2143 - loss: 2.8326
Epoch 7/200
1/1 - 0s - 46ms/step - accuracy: 0.2143 - loss: 2.8236
Epoch 8/200
1/1 - 0s - 47ms/step - accuracy: 0.2857 - loss: 2.8144
Epoch 9/200
1/1 - 0s - 47ms/step - accuracy: 0.2857 - loss: 2.8049
Epoch 10/200
1/1 - 0s - 47ms/step - accuracy: 0.2857 - loss: 2.7952
Epoch 11/200
1/1 - 0s - 47ms/step - accuracy: 0.2857 - loss: 2.7852
Epoch 12/200
1/1 - 0s - 50ms/step - accuracy: 0.2857 - loss: 2.7748
Epoch 13/200
1/1 - 0s - 47ms/step - accuracy: 0.2857 - loss: 2.7640
Epoch 14/200
1/1 - 0s - 47ms/step - accuracy: 0.2857 - loss: 2.7528
Epoch 15/200
1/1 - 0s - 48ms/step - accuracy: 0.2857 - los

In [16]:
# 9. 문장 생성 함수


# 인덱스 → 단어 변환 딕셔너리
index_to_word = {}

for word, index in t.word_index.items():
    index_to_word[index] = word


def sentence_generation(model, tokenizer, current_word, n):
    init_word = current_word
    sentence = ""

    for _ in range(n):
        encoded = tokenizer.texts_to_sequences([current_word])[0]

        encoded = pad_sequences(
            [encoded],
            maxlen=max_len - 1,
            padding="pre"
        )

        prediction = model.predict(encoded, verbose=0)
        predicted_index = np.argmax(prediction, axis=1)[0]

        predicted_word = index_to_word[predicted_index]

        current_word = current_word + " " + predicted_word
        sentence = sentence + " " + predicted_word

    sentence = init_word + sentence

    return sentence

In [17]:

# 10. 문장 생성 테스트

print(sentence_generation(model, t, "고기는", 5))
print(sentence_generation(model, t, "낫말은", 5))
print(sentence_generation(model, t, "구슬이", 4))

고기는 씹어야 맛이고 말은 해야 맛이라
낫말은 새가 듣고 밤말은 쥐가 듣는다
구슬이 서 말이라도 꿰어야 보배
